In [7]:
import requests

# Test 1: check position endpoint
r = requests.get("http://localhost:9005/data/current/position")
print("Position:", r.json())

# Test 2: try a single known-good location
r = requests.get("http://localhost:9005/data/image/sentinel", params={
    "lat": 48.8566,
    "lon": 2.3522,
    "timestamp": "2026-01-01T12:00:00Z",
    "spectral_bands": "red,green,blue",
    "size_km": 10.0,
    "return_type": "png",
    "window_seconds": 864000,
})
print("Status:", r.status_code)
print("Body:", r.text)

Position: {'lon-lat-alt': [159.89107108950208, 16.822739677143627, 792.0049079080574], 'timestamp': '2026-01-01T00:01:00Z'}
Status: 500
Body: Internal Server Error


In [9]:
r = requests.get("http://localhost:9005/data/image/sentinel", params={
    "lat": 48.8566,
    "lon": 2.3522,
    "timestamp": "2026-01-01T12:00:00Z",
    "spectral_bands": ["red", "green", "blue"],  # ← list, not string
    "size_km": 10.0,
    "return_type": "png",
    "window_seconds": 864000,
})
print("Status:", r.status_code)

Status: 200


In [1]:
import requests, json

r = requests.get("http://localhost:9005/data/image/sentinel", params={
    "lat": 48.8566,
    "lon": 2.3522,
    "timestamp": "2026-01-01T12:00:00Z",
    "spectral_bands": ["red", "green", "blue"],
    "size_km": 10.0,
    "return_type": "png",
    "window_seconds": 864000,
})

print("Status      :", r.status_code)
print("Content-Type:", r.headers.get("Content-Type"))
print("X-Metadata  :", r.headers.get("X-Image-Metadata"))
print("Body preview:", r.content[:300])  # first 300 bytes

Status      : 200
Content-Type: image/png
X-Metadata  : None
Body preview: b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x03\xf4\x00\x00\x03\xf2\x08\x02\x00\x00\x00]\x1e\x03(\x00\x01\x00\x00IDATx\x9c\xec\xfdYs\x9cG\xa2\xa7y\xfe}}\x97\x88\x00@\x8a\x922\xf3TVWWY\xf7\xdc\xb4\xcd\x98\xcdX\x7f\xffO\xd0s\xd1fc\xd3\xd3\xd3=VU\xa7N.ZH\x00\xb1\xbc\x8bosA\xea\xa4tRJI\x14\xc0\x00\x02\xcf\xef"Mb\x8a\x80\x07\x11 \x9e\xf0\xf0\xc5\xb4\xd6\x04\x00\x00\x00<GY\xb7Y\xbe\x97K\xf9x\xa7C\xf5\xc3\x17\xfa\xf2\xdc\x83:#\x7f\xee\x01\x00\x00\x00\x00\x1f\xcb(\xac\x9af\x15\x93j\xb6\xd6\xf9\x17>om\xcf=\x00\x00\x00\x00\xe0c9\x05\x15\x9bk[kQ\x8dV\xdb|\xee!\x9d\x15q\x0f\x00\x00\x80g,\x8e.\xdab\x9bQ\xb0v\xab\xe1e/Ly\xd9\x8f\x1e\x00\x00\x00\xcf\x9d\x97\x1dM79\xb5n\xec\xe4\xce=\x9c\xf32l\xa8\x05\x00\x00\xc0\xb3\x97\x94\xac\xac#\xee\x89{\x00\x00\x00\xe0"\xb0\xe6\x1e\x00\x00\x00\xb8\x10\xc4=\x00\x00\x00\xf0\xe8\x16iy\xfc\xcf\xc2\x86Z\x00\x00\x00\xe0q\x9d\x92V\xa9\x05e\xc9H\xe3\xa3'


In [2]:
import requests

r = requests.get("http://localhost:9005/data/image/sentinel", params={
    "lat": 48.8566,
    "lon": 2.3522,
    "timestamp": "2026-01-01T12:00:00Z",
    "spectral_bands": ["red", "green", "blue"],
    "size_km": 10.0,
    "return_type": "png",
    "window_seconds": 864000,
})

print("All headers:")
for k, v in r.headers.items():
    print(f"  {k}: {v}")

All headers:
  date: Thu, 26 Mar 2026 13:28:13 GMT
  server: uvicorn
  sentinel_metadata: {"image_available": true, "source": "sentinel-2a", "spectral_bands": ["red", "green", "blue"], "footprint": [2.2838568688068626, 48.811633919704065, 2.420543131193137, 48.901566080295936], "size_km": 10.0, "cloud_cover": 99.980474, "datetime": "2026-01-01T10:57:49Z"}
  access-control-expose-headers: sentinel_metadata
  content-length: 86521
  content-type: image/png


In [3]:
"""
Cloud Cover Dataset Generator for LFM2-VL Fine-tuning
======================================================
Uses the SimSat API (localhost:9005) to collect Sentinel-2 images
with ground-truth cloud cover labels.

Output structure:
  dataset/
    images/          ← PNG files named by location+timestamp
    metadata.jsonl   ← one JSON record per image (for fine-tuning)
    dataset.json     ← full dataset manifest

Usage:
  1. Start SimSat:  docker compose up
  2. Run:           python generate_dataset.py

Adjust CONFIG below to control how many samples you want.
"""

import os
import json
import time
import random
import requests
import hashlib
from datetime import datetime, timedelta, timezone
from pathlib import Path

# ─────────────────────────────────────────────
# CONFIG — tweak these to your needs
# ─────────────────────────────────────────────
CONFIG = {
    "api_base": "http://localhost:9005",
    "output_dir": "dataset",
    "num_samples": 500,          # total images to collect
    "size_km": 10.0,             # image footprint size in km
    "window_seconds": 864000,    # 10-day lookback for Sentinel images
    "spectral_bands": ["red", "green", "blue"],  # change to add NIR, SWIR etc.
    "delay_seconds": 1.5,        # be polite to the (slow) Sentinel API
    "skip_ocean": True,          # skip positions where no image is available
    "min_valid_images": 300,     # stop early only if we hit this many valid ones

    # Stratified sampling: roughly equal buckets across cloud cover %
    # Set to None to skip stratification (just collect whatever)
    "cloud_buckets": {
        "clear":       (0,   20),   # 0–20% cloud
        "partly":      (20,  60),   # 20–60% cloud
        "overcast":    (60, 100),   # 60–100% cloud
    },
}

# ─────────────────────────────────────────────
# Geographic sampling — land-biased coordinates
# (avoids mostly-ocean positions)
# ─────────────────────────────────────────────
LAND_REGIONS = [
    # (lat_min, lat_max, lon_min, lon_max, label)
    (35,  70,  -10,  40,  "europe"),
    (25,  50,  -125, -65, "north_america"),
    (-35, 35,  -80,  55,  "africa_south_america"),
    (20,  55,   60, 140,  "asia"),
    (-45, -10, 110, 155,  "australia"),
]

def sample_land_coordinate():
    region = random.choice(LAND_REGIONS)
    lat_min, lat_max, lon_min, lon_max, _ = region
    lat = random.uniform(lat_min, lat_max)
    lon = random.uniform(lon_min, lon_max)
    return round(lat, 6), round(lon, 6)

def sample_timestamp(days_back=365):
    """Random timestamp within the past N days."""
    now = datetime.now(timezone.utc)
    delta = timedelta(seconds=random.randint(0, days_back * 86400))
    ts = now - delta
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")

# ─────────────────────────────────────────────
# VQA prompt templates for fine-tuning
# ─────────────────────────────────────────────
def build_vqa_record(image_filename, meta):
    """
    Builds a VQA-style record suitable for LFM2-VL fine-tuning.
    We generate multiple QA pairs per image for richer training signal.
    """
    cloud_pct = round(meta["cloud_cover"], 1)

    # Bucket label
    if cloud_pct < 20:
        bucket = "clear"
        bucket_desc = "mostly clear skies with little to no cloud cover"
    elif cloud_pct < 60:
        bucket = "partly_cloudy"
        bucket_desc = "partial cloud cover"
    else:
        bucket = "overcast"
        bucket_desc = "heavy cloud cover"

    qa_pairs = [
        {
            "question": "What percentage of this satellite image is covered by clouds?",
            "answer": f"{cloud_pct}%"
        },
        {
            "question": "Classify the cloud cover in this image as: clear, partly cloudy, or overcast.",
            "answer": bucket.replace("_", " ")
        },
        {
            "question": "Describe the sky conditions visible in this satellite image.",
            "answer": f"The image shows {bucket_desc}, with approximately {cloud_pct}% of the area covered by clouds."
        },
        {
            "question": "Is this satellite image suitable for ground surface analysis?",
            "answer": (
                "Yes, the image is suitable for ground analysis as cloud cover is low."
                if cloud_pct < 20 else
                "Partially. Some ground features may be obscured by cloud cover."
                if cloud_pct < 60 else
                "No, heavy cloud cover significantly obscures the ground surface."
            )
        },
    ]

    return {
        "image": image_filename,
        "cloud_cover_pct": cloud_pct,
        "cloud_bucket": bucket,
        "source_satellite": meta.get("source", "unknown"),
        "capture_datetime": meta.get("datetime", ""),
        "footprint": meta.get("footprint", []),
        "size_km": meta.get("size_km", CONFIG["size_km"]),
        "spectral_bands": meta.get("spectral_bands", ["red", "green", "blue"]),
        "qa_pairs": qa_pairs,
        # Convenience field: single instruction-following format
        "instruction": "Analyze this satellite image and estimate the cloud cover percentage.",
        "response": f"This satellite image has approximately {cloud_pct}% cloud cover ({bucket.replace('_', ' ')})."
    }

# ─────────────────────────────────────────────
# API call
# ─────────────────────────────────────────────
def fetch_sentinel_image(lat, lon, timestamp, output_path):
    """
    Calls /data/image/sentinel and saves the PNG.
    Returns the metadata dict, or None if unavailable.
    """
    params = {
        "lat": lat,
        "lon": lon,
        "timestamp": timestamp,
        "spectral_bands": CONFIG["spectral_bands"],  # now a list, requests handles it correctly
        "size_km": CONFIG["size_km"],
        "return_type": "png",
        "window_seconds": CONFIG["window_seconds"],
    }

    try:
        resp = requests.get(
            f"{CONFIG['api_base']}/data/image/sentinel",
            params=params,
            timeout=60,
            stream=True,
        )
        resp.raise_for_status()

        # Metadata comes in response headers (SimSat convention)
        meta_header = resp.headers.get("sentinel_metadata")
        if meta_header:
            meta = json.loads(meta_header)
        else:
            # Fall back: parse from response if JSON, otherwise treat as image
            content_type = resp.headers.get("Content-Type", "")
            if "application/json" in content_type:
                body = resp.json()
                if not body.get("image_available", True):
                    return None
                meta = body
            else:
                # Pure image response — minimal metadata
                meta = {
                    "image_available": True,
                    "cloud_cover": -1,  # unknown
                    "source": "sentinel-2",
                }

        if not meta.get("image_available", True):
            return None
        if meta.get("cloud_cover", -1) == -1:
            return None  # can't label without cloud cover

        # Save image
        with open(output_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)

        return meta

    except requests.exceptions.RequestException as e:
        print(f"  ⚠ Request failed: {e}")
        return None

# ─────────────────────────────────────────────
# Alternate approach: use /data/current endpoints
# (when simulation is running)
# ─────────────────────────────────────────────
def fetch_current_sentinel_image(output_path):
    """
    Fetches image from the CURRENT satellite position.
    Use this if you're running the simulation live.
    """
    params = {
        "spectral_bands": CONFIG["spectral_bands"],
        "size_km": CONFIG["size_km"],
        "return_type": "png",
        "window_seconds": CONFIG["window_seconds"],
    }
    try:
        resp = requests.get(
            f"{CONFIG['api_base']}/data/current/image/sentinel",
            params=params,
            timeout=60,
            stream=True,
        )
        resp.raise_for_status()

        meta_header = resp.headers.get("sentinel_metadata")
        if meta_header:
            meta = json.loads(meta_header)
        else:
            content_type = resp.headers.get("Content-Type", "")
            if "application/json" in content_type:
                meta = resp.json()
            else:
                meta = {"image_available": True, "cloud_cover": -1}

        if not meta.get("image_available", True) or meta.get("cloud_cover", -1) == -1:
            return None, None

        pos = meta.get("satellite_position", [0, 0, 0])
        ts = meta.get("timestamp", "unknown")

        with open(output_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)

        return meta, (pos[1], pos[0])  # lat, lon

    except requests.exceptions.RequestException as e:
        print(f"  ⚠ Request failed: {e}")
        return None, None

# ─────────────────────────────────────────────
# Main generation loop
# ─────────────────────────────────────────────
def main():
    out_dir = Path(CONFIG["output_dir"])
    img_dir = out_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)

    records = []
    bucket_counts = {b: 0 for b in CONFIG["cloud_buckets"]}
    attempts = 0
    valid = 0
    target = CONFIG["num_samples"]

    print(f"\n🛰  SimSat Cloud Cover Dataset Generator")
    print(f"   Target: {target} images → {img_dir}\n")

    while valid < target and attempts < target * 5:
        attempts += 1
        lat, lon = sample_land_coordinate()
        timestamp = sample_timestamp(days_back=365)

        # Unique filename based on coords + timestamp
        uid = hashlib.md5(f"{lat}{lon}{timestamp}".encode()).hexdigest()[:10]
        img_filename = f"sentinel_{uid}.png"
        img_path = img_dir / img_filename

        print(f"[{valid+1}/{target}] lat={lat:.3f} lon={lon:.3f} ts={timestamp[:10]}  ", end="", flush=True)

        meta = fetch_sentinel_image(lat, lon, timestamp, img_path)

        if meta is None:
            print("✗ no image")
            time.sleep(CONFIG["delay_seconds"])
            continue

        cloud_pct = meta.get("cloud_cover", -1)
        if cloud_pct < 0:
            print("✗ no cloud label")
            img_path.unlink(missing_ok=True)
            time.sleep(CONFIG["delay_seconds"])
            continue

        # Stratification check
        assigned_bucket = None
        for bucket_name, (lo, hi) in CONFIG["cloud_buckets"].items():
            if lo <= cloud_pct < hi:
                assigned_bucket = bucket_name
                break
        if assigned_bucket is None:
            assigned_bucket = "overcast"  # catch 100%

        # Skip if this bucket is already full (stratified sampling)
        per_bucket_target = target // len(CONFIG["cloud_buckets"])
        if bucket_counts[assigned_bucket] >= per_bucket_target * 1.5:
            print(f"✗ bucket '{assigned_bucket}' full ({bucket_counts[assigned_bucket]})")
            img_path.unlink(missing_ok=True)
            time.sleep(CONFIG["delay_seconds"] / 2)
            continue

        record = build_vqa_record(img_filename, meta)
        records.append(record)
        bucket_counts[assigned_bucket] = bucket_counts.get(assigned_bucket, 0) + 1
        valid += 1

        print(f"✓  cloud={cloud_pct:.1f}%  [{assigned_bucket}]  (buckets: {bucket_counts})")

        # Write metadata incrementally (safe to interrupt)
        with open(out_dir / "metadata.jsonl", "a") as f:
            f.write(json.dumps(record) + "\n")

        time.sleep(CONFIG["delay_seconds"])

    # ── Write final manifest ──
    manifest = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "total_images": len(records),
        "bucket_distribution": bucket_counts,
        "config": CONFIG,
        "records": records,
    }
    with open(out_dir / "dataset.json", "w") as f:
        json.dump(manifest, f, indent=2)

    # ── Write train/val split ──
    random.shuffle(records)
    split_idx = int(len(records) * 0.85)
    train_records = records[:split_idx]
    val_records = records[split_idx:]

    with open(out_dir / "train.jsonl", "w") as f:
        for r in train_records:
            f.write(json.dumps(r) + "\n")

    with open(out_dir / "val.jsonl", "w") as f:
        for r in val_records:
            f.write(json.dumps(r) + "\n")

    print(f"\n✅ Done!")
    print(f"   Total collected : {len(records)}")
    print(f"   Bucket breakdown: {bucket_counts}")
    print(f"   Train / Val     : {len(train_records)} / {len(val_records)}")
    print(f"   Output dir      : {out_dir.resolve()}")
    print(f"\n   Files:")
    print(f"     dataset/images/     ← PNG satellite images")
    print(f"     dataset/metadata.jsonl  ← incremental records")
    print(f"     dataset/dataset.json    ← full manifest")
    print(f"     dataset/train.jsonl     ← 85% training split")
    print(f"     dataset/val.jsonl       ← 15% validation split")


if __name__ == "__main__":
    main()


🛰  SimSat Cloud Cover Dataset Generator
   Target: 500 images → dataset\images

[1/500] lat=55.799 lon=20.885 ts=2025-09-15  ✓  cloud=84.0%  [overcast]  (buckets: {'clear': 0, 'partly': 0, 'overcast': 1})
[2/500] lat=-28.164 lon=132.901 ts=2025-10-09  ✓  cloud=0.1%  [clear]  (buckets: {'clear': 1, 'partly': 0, 'overcast': 1})
[3/500] lat=33.134 lon=124.330 ts=2025-06-18  ✓  cloud=85.2%  [overcast]  (buckets: {'clear': 1, 'partly': 0, 'overcast': 2})
[4/500] lat=42.786 lon=-88.131 ts=2025-10-26  ✓  cloud=22.0%  [partly]  (buckets: {'clear': 1, 'partly': 1, 'overcast': 2})
[5/500] lat=21.079 lon=102.297 ts=2026-03-14  ✓  cloud=92.4%  [overcast]  (buckets: {'clear': 1, 'partly': 1, 'overcast': 3})
[6/500] lat=-19.179 lon=130.788 ts=2025-06-24  ✓  cloud=0.0%  [clear]  (buckets: {'clear': 2, 'partly': 1, 'overcast': 3})
[7/500] lat=31.649 lon=-87.261 ts=2025-03-27  ✓  cloud=0.0%  [clear]  (buckets: {'clear': 3, 'partly': 1, 'overcast': 3})
[8/500] lat=66.110 lon=2.045 ts=2025-12-24  ✗ no i

In [4]:
import shutil
shutil.make_archive("cloud_dataset", "zip", "dataset")

'C:\\Users\\desk-7\\Music\\AI in Space\\cloud_dataset.zip'

In [5]:
from PIL import Image
import numpy as np
import json
from pathlib import Path

DATASET_DIR = Path("dataset")
THRESHOLD = 0.15  # if >15% of pixels are pure black, reject image

def is_corrupted(img_path, threshold=THRESHOLD):
    img = np.array(Image.open(img_path).convert("RGB"))
    black_pixels = np.all(img < 10, axis=-1).sum()
    total_pixels = img.shape[0] * img.shape[1]
    return (black_pixels / total_pixels) > threshold

# Scan all images
print("Scanning dataset for corrupted images...")
corrupted = []
clean = []

for split in ["train.jsonl", "val.jsonl"]:
    with open(DATASET_DIR / split) as f:
        records = [json.loads(l) for l in f]
    
    clean_records = []
    for r in records:
        img_path = DATASET_DIR / "images" / r["image"]
        if is_corrupted(img_path):
            corrupted.append(r["image"])
        else:
            clean_records.append(r)
    
    # Overwrite with clean version
    with open(DATASET_DIR / split, "w") as f:
        for r in clean_records:
            f.write(json.dumps(r) + "\n")
    
    print(f"  {split}: {len(records)} → {len(clean_records)} records")

print(f"\n🗑️  Removed {len(corrupted)} corrupted images")
print(f"✅  Clean dataset ready for next training run")

Scanning dataset for corrupted images...
  train.jsonl: 425 → 331 records
  val.jsonl: 75 → 63 records

🗑️  Removed 106 corrupted images
✅  Clean dataset ready for next training run


In [6]:
import shutil, zipfile, os
from pathlib import Path

DATASET_DIR = Path("dataset")

# Read all image filenames still in the clean jsonl files
import json

clean_images = set()
for split in ["train.jsonl", "val.jsonl"]:
    with open(DATASET_DIR / split) as f:
        for line in f:
            record = json.loads(line)
            clean_images.add(record["image"])

print(f"Clean images to include: {len(clean_images)}")

# Create zip with only clean images + jsonl files
ZIP_PATH = "cloud_dataset_clean.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # Add jsonl files
    for f in ["train.jsonl", "val.jsonl", "metadata.jsonl", "dataset.json"]:
        path = DATASET_DIR / f
        if path.exists():
            zf.write(path, f"dataset/{f}")
    
    # Add only clean images
    for img_name in clean_images:
        img_path = DATASET_DIR / "images" / img_name
        if img_path.exists():
            zf.write(img_path, f"dataset/images/{img_name}")

size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f"✅ Created {ZIP_PATH} ({size_mb:.1f} MB)")
print(f"   Upload this to Google Drive and update ZIP_PATH in Colab to 'cloud_dataset_clean.zip'")

Clean images to include: 394
✅ Created cloud_dataset_clean.zip (406.5 MB)
   Upload this to Google Drive and update ZIP_PATH in Colab to 'cloud_dataset_clean.zip'


In [7]:
import json
from pathlib import Path
from collections import Counter

records = []
for split in ["train.jsonl", "val.jsonl"]:
    with open(Path("dataset") / split) as f:
        records += [json.loads(l) for l in f]

buckets = Counter(r["cloud_bucket"] for r in records)
print("Bucket distribution:")
for b, count in buckets.items():
    bar = "█" * (count // 5)
    print(f"  {b:<15} {count:>4}  {bar}")

Bucket distribution:
  overcast         162  ████████████████████████████████
  clear            152  ██████████████████████████████
  partly_cloudy     80  ████████████████
